In [2]:
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

# Target
target = "satisfaction"

# Features
drop_cols = [target, "id"]

X = train.drop(columns=drop_cols).copy()
y = train[target].astype(int)

X_test = test.drop(columns=["id"]).copy()

# Categorical columns
cat_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

# Same validation split for every model
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)
print("Positive rate:", y.mean())

Train: (559708, 21)
Valid: (139927, 21)
Test : (299844, 21)
Positive rate: 0.44357272006117476


===============================================**CATBOOST**=======================

In [4]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.08,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=100,
    allow_writing_files=False,
    thread_count=-1
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100
)

0:	test: 0.9438315	best: 0.9438315 (0)	total: 540ms	remaining: 13m 30s
100:	test: 0.9560816	best: 0.9560816 (100)	total: 37s	remaining: 8m 32s
200:	test: 0.9569555	best: 0.9569555 (200)	total: 1m 13s	remaining: 7m 52s
300:	test: 0.9573263	best: 0.9573271 (299)	total: 1m 48s	remaining: 7m 11s
400:	test: 0.9574382	best: 0.9574382 (400)	total: 2m 21s	remaining: 6m 27s
500:	test: 0.9575074	best: 0.9575109 (496)	total: 2m 57s	remaining: 5m 54s
600:	test: 0.9575823	best: 0.9575823 (600)	total: 3m 32s	remaining: 5m 17s
700:	test: 0.9576380	best: 0.9576425 (697)	total: 4m 7s	remaining: 4m 42s
800:	test: 0.9576902	best: 0.9576977 (786)	total: 4m 44s	remaining: 4m 8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9576977387
bestIteration = 786

Shrink model to first 787 iterations.


CatBoostClassifier(allow_writing_files=False, depth=8, eval_metric='AUC', iterations=1500, learning_rate=0.08, loss_function='Logloss', random_seed=42, verbose=100)

In [5]:
cat_valid_pred = cat_model.predict_proba(X_valid)[:, 1]
cat_test_pred = cat_model.predict_proba(X_test)[:, 1]

cat_auc = roc_auc_score(y_valid, cat_valid_pred)

print(f"CatBoost AUC: {cat_auc:.8f}")

CatBoost AUC: 0.95769774


In [6]:
os.makedirs("artifacts/oof", exist_ok=True)
os.makedirs("artifacts/test", exist_ok=True)

np.save(
    "artifacts/oof/baseline_cat.npy",
    cat_valid_pred
)

np.save(
    "artifacts/test/baseline_cat.npy",
    cat_test_pred
)

In [7]:
cat_model.save_model("artifacts/baseline_catboost.cbm")

======================================**Light LGM**==================================================

In [8]:
import lightgbm as lgb

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_valid_lgb[col] = X_valid_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].astype("category")

In [9]:
lgb_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=SEED,
    n_jobs=-1
)
lgb_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=cat_cols,
    eval_set=[(X_valid_lgb, y_valid)],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)

C:\Users\Garv\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248271, number of negative: 311437
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031350 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 597
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443572 -> initscore=-0.226676
[LightGBM] [Info] Start training from score -0.226676
Training until validation scores don't improve for 100 rounds
[100]	valid_0's binary_logloss: 0.242808
[200]	valid_0's binary_logloss: 0.231924
[300]	valid_0's binary_logloss: 0.230565
[400]	valid_0's binary_logloss: 0.230227
[500]	valid_0's binary_logloss: 0.23007
[600]	valid_0'

LGBMClassifier(colsample_bytree=0.85, learning_rate=0.03, n_estimators=3000,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               reg_alpha=0.1, reg_lambda=0.5, subsample=0.85)

In [10]:
lgb_valid_pred = lgb_model.predict_proba(X_valid_lgb)[:, 1]
lgb_test_pred = lgb_model.predict_proba(X_test_lgb)[:, 1]

lgb_auc = roc_auc_score(y_valid, lgb_valid_pred)

print(f"LightGBM AUC: {lgb_auc:.8f}")

LightGBM AUC: 0.95787464


In [11]:
np.save(
    "artifacts/oof/baseline_lgb.npy",
    lgb_valid_pred
)

np.save(
    "artifacts/test/baseline_lgb.npy",
    lgb_test_pred
)

lgb.booster = lgb_model.booster_
lgb_model.booster_.save_model(
    "artifacts/baseline_lightgbm.txt"
)

==============================================**XG BOOST**========================================

In [12]:
import xgboost as xgb

X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()
X_test_xgb = X_test.copy()

for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_valid_xgb[col] = X_valid_xgb[col].astype("category")
    X_test_xgb[col] = X_test_xgb[col].astype("category")

In [15]:
xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    random_state=SEED,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_valid_xgb, y_valid)],
    verbose=100
)

xgb_valid_pred = xgb_model.predict_proba(X_valid_xgb)[:, 1]
xgb_test_pred = xgb_model.predict_proba(X_test_xgb)[:, 1]

xgb_auc = roc_auc_score(y_valid, xgb_valid_pred)

print(f"XGBoost AUC: {xgb_auc:.8f}")

[0]	validation_0-auc:0.94990
[100]	validation_0-auc:0.95532
[200]	validation_0-auc:0.95663
[300]	validation_0-auc:0.95741
[400]	validation_0-auc:0.95769
[500]	validation_0-auc:0.95785
[600]	validation_0-auc:0.95795
[700]	validation_0-auc:0.95798
[800]	validation_0-auc:0.95801
[900]	validation_0-auc:0.95799
[1000]	validation_0-auc:0.95798
[1100]	validation_0-auc:0.95797
[1200]	validation_0-auc:0.95795
[1300]	validation_0-auc:0.95791
[1400]	validation_0-auc:0.95784
[1499]	validation_0-auc:0.95780
XGBoost AUC: 0.95779618


In [16]:
np.save(
    "artifacts/oof/baseline_xgb.npy",
    xgb_valid_pred
)

np.save(
    "artifacts/test/baseline_xgb.npy",
    xgb_test_pred
)

xgb_model.save_model(
    "artifacts/baseline_xgboost.json"
)

==================================**RESULTS**=================================

In [17]:
results = pd.DataFrame({
    "Model": [
        "CatBoost",
        "LightGBM",
        "XGBoost"
    ],
    "AUC": [
        cat_auc,
        lgb_auc,
        xgb_auc
    ]
}).sort_values("AUC", ascending=False)

print(results)

      Model       AUC
1  LightGBM  0.957875
2   XGBoost  0.957796
0  CatBoost  0.957698


In [22]:
pred_df = pd.DataFrame({
    "CatBoost": cat_valid_pred,
    "LightGBM": lgb_valid_pred,
    "XGBoost": xgb_valid_pred
})

print("\nPrediction correlations:")
print(pred_df.corr())


Prediction correlations:
          CatBoost  LightGBM   XGBoost
CatBoost  1.000000  0.997972  0.997875
LightGBM  0.997972  1.000000  0.997846
XGBoost   0.997875  0.997846  1.000000


In [21]:
blend_pred = (
    0.333333 * cat_valid_pred +
    0.333333 * lgb_valid_pred +
    0.333334 * xgb_valid_pred
)

blend_auc = roc_auc_score(y_valid, blend_pred)

print(f"\nSimple 3-model blend AUC: {blend_auc:.8f}")


Simple 3-model blend AUC: 0.95822617
